# 05 — Stage 1: Ensemble Training

Train and cross-validate the ML super-learner ensemble for predicting third-party
military intervention.

**Inputs**: `data/interim/dd_spat_{cy}_{ud}.parquet` (25 files from notebook 04)

**Outputs**:
- `data/interim/sl_oof_{cy}_{ud}.parquet` — out-of-fold probability predictions
  for every onset dyad-year (25 files)
- `data/interim/sl_weights_{cy}_{ud}.parquet` — NNLS ensemble weights per imputation
- `results/tables/tab-tuning-by-model.tex` — component model performance

**Reference R scripts**: `14-trainModels.R`, `13-makePRcomp.R`, `16-makePirate.R`

**Pipeline per imputation dataset**:
1. Extract onset rows with coded intervention status; build feature matrix
2. PCA: retain components to scree-plot elbow
3. 10-fold CV, stratified by onset (leave-one-conflict-out where possible):
   - Random forest, elastic net, multinomial logit, MLP
4. NNLS super learner: minimise held-out log-loss
5. Save out-of-fold predictions and ensemble weights

**Outcome variable**: `intervention` — 0 (none), 1 (gov-biased), 2 (opp-biased)

**Training sample**: All onset dyad-years with coded intervention status
(Regan 1944–1999 + post-1999 hand-coded onsets). Approximately 25,700 DD rows
per imputation (up from ~19,700 with Regan-only).

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import nnls
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, roc_auc_score
import joblib

sys.path.insert(0, str(Path("..").resolve() / "src"))
from shadow.data.spatial import update_spatial_lags_proba, _build_W_cache, ALL_SPAT_COLS

warnings.filterwarnings("ignore")

ROOT    = Path("..").resolve()
INTERIM = ROOT / "data" / "interim"
RESULTS = ROOT / "results"
(RESULTS / "tables").mkdir(parents=True, exist_ok=True)

N_FOLDS    = 10
MAX_FP_ITER = 5      # maximum fixed-point iterations
FP_TOL     = 1e-4   # convergence: mean |Δ spat_gov| + |Δ spat_opp|
SEED       = 90210

## § 1  Feature & outcome columns

In [2]:
# Columns that identify a row — not used as features.
# NOTE: `year` is deliberately excluded from this list so it enters
# the feature matrix, letting trees learn temporal breakpoints.
ID_COLS = [
    "ccode_A", "ccode_B", "ddyear",
    "onset_A", "regan_period", "intervention",
]

# ── Feature-set channels ─────────────────────────────────────────────────
# Each classifier is trained on three feature sets; NNLS picks weights
# across all (classifier × feature-set) combinations jointly.
#   X  = base dyad characteristics only (no spatial lags)
#   W  = spatial lags only
#   XW = everything (base + spatial)
FEATURE_MODES = ["X", "W", "XW"]


def get_feature_cols(df: pd.DataFrame, mode: str = "XW") -> list[str]:
    """Return numeric feature columns for the given ablation mode."""
    exclude = set(ID_COLS)
    all_feat = [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]
    spat_set = set(ALL_SPAT_COLS)
    if mode == "X":
        return [c for c in all_feat if c not in spat_set]
    elif mode == "W":
        return [c for c in all_feat if c in spat_set]
    else:  # XW
        return all_feat


def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add year-based temporal features for the classifier."""
    df = df.copy()
    df["cold_war"] = (df["year"] <= 1990).astype(int)
    return df


def prl(y_true: np.ndarray, proba: np.ndarray) -> float:
    """Proportional Reduction in Loss vs. class-frequency null."""
    null_p = np.bincount(y_true.astype(int), minlength=3) / len(y_true)
    null_p = np.clip(null_p, 1e-9, None)
    null_loss = -np.log(null_p[y_true.astype(int)]).mean()
    model_loss = log_loss(y_true, proba, labels=[0, 1, 2])
    return (null_loss - model_loss) / null_loss

## § 2  Super-learner training function

One unified ensemble across all (classifier × feature-set) combinations.
Each feature mode (X, W, XW) gets its own PCA; all 27 candidate models
feed into a single NNLS stacking layer that picks weights jointly.

In [3]:
def make_classifiers(seed: int) -> dict:
    """
    Instantiate component classifiers spanning hyperparameter space.

    Nine candidates across three families:
      Trees     — random forest; HGB at two learning rates
      Logistic  — ridge, elastic net (l1_ratio=0.5), lasso, unpenalised
      Neural    — MLP small (25,) and large (100, 50)

    NNLS stacking selects the best convex combination; adding weak candidates
    carries no penalty (they receive zero weight).
    """
    return {
        # ── Tree ensembles ──────────────────────────────────────────────
        "rf":       RandomForestClassifier(
                        n_estimators=500, max_features="sqrt",
                        n_jobs=-1, random_state=seed),
        "hgb":      HistGradientBoostingClassifier(
                        learning_rate=0.1, max_iter=300,
                        early_stopping=True, validation_fraction=0.1,
                        n_iter_no_change=15, random_state=seed),
        "hgb_lo":   HistGradientBoostingClassifier(
                        learning_rate=0.05, max_iter=500,
                        early_stopping=True, validation_fraction=0.1,
                        n_iter_no_change=15, random_state=seed),
        # ── Penalised / unpenalised logistic ────────────────────────────
        "ridge":    LogisticRegression(
                        penalty="l2", solver="lbfgs", C=1.0,
                        max_iter=2000, random_state=seed),
        "glmnet":   LogisticRegression(
                        penalty="elasticnet",
                        solver="saga", l1_ratio=0.5, C=1.0,
                        max_iter=2000, random_state=seed),
        "lasso":    LogisticRegression(
                        penalty="l1", solver="saga", C=1.0,
                        max_iter=2000, random_state=seed),
        "multinom": LogisticRegression(
                        penalty=None, solver="lbfgs",
                        max_iter=2000, random_state=seed),
        # ── Neural networks ─────────────────────────────────────────────
        "mlp_sm":   MLPClassifier(
                        hidden_layer_sizes=(25,), max_iter=1000,
                        early_stopping=True, random_state=seed),
        "mlp_lg":   MLPClassifier(
                        hidden_layer_sizes=(100, 50), max_iter=1000,
                        early_stopping=True, random_state=seed),
    }


def _predict_proba_3class(clf, X_val: np.ndarray) -> np.ndarray:
    """Return (n, 3) probability array aligned to classes [0, 1, 2]."""
    raw = clf.predict_proba(X_val)
    out = np.zeros((len(X_val), 3))
    for j, cls in enumerate(clf.classes_):
        out[:, int(cls)] = raw[:, j]
    return out


def _fit_pca(X: np.ndarray, seed: int) -> tuple[StandardScaler, PCA, np.ndarray]:
    """Scale, fit PCA to 90% variance, return (scaler, pca, X_pc)."""
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)
    pca_full = PCA(random_state=seed).fit(X_sc)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    n_comp = max(5, min(int(np.searchsorted(cumvar, 0.90)) + 1, X_sc.shape[1] - 1))
    pca = PCA(n_components=n_comp, random_state=seed).fit(X_sc)
    return scaler, pca, pca.transform(X_sc)


def train_super_learner(
    df: pd.DataFrame,
    y: np.ndarray,
    seed: int = SEED,
) -> dict:
    """
    Train a unified NNLS super-learner across all (classifier × feature-set)
    combinations via 10-fold CV.

    Each feature mode (X, W, XW) gets its own PCA pipeline. All 27 candidate
    models (9 classifiers × 3 feature sets) are stacked jointly — NNLS
    determines which (method, feature-set) pairs earn weight.

    Returns a dict with keys:
      'oof_proba'        : (n, 3) ensemble OOF probabilities
      'weights'          : dict {(mode, name): float}
      'pipelines'        : dict {mode: {'scaler', 'pca', 'n_pca'}}
      'classifiers'      : dict {(mode, name): fitted clf}
      'component_metrics': DataFrame with per-component CV metrics
    """
    # ── Per-mode PCA pipelines ────────────────────────────────────────────
    pipelines = {}
    X_pcs = {}
    for mode in FEATURE_MODES:
        feat_cols = get_feature_cols(df, mode=mode)
        X_raw = df[feat_cols].fillna(0).to_numpy(dtype=float)
        scaler, pca, X_pc = _fit_pca(X_raw, seed)
        pipelines[mode] = {"scaler": scaler, "pca": pca,
                           "n_pca": pca.n_components_, "feat_cols": feat_cols}
        X_pcs[mode] = X_pc

    # ── 10-fold CV across all (mode, classifier) pairs ────────────────────
    clf_templates = make_classifiers(seed)
    keys = [(mode, name) for mode in FEATURE_MODES for name in clf_templates]

    oof: dict[tuple, np.ndarray] = {
        k: np.zeros((len(y), 3)) for k in keys
    }

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X_pcs["XW"], y)):
        y_tr = y[tr_idx]
        for mode in FEATURE_MODES:
            X_tr = X_pcs[mode][tr_idx]
            X_val = X_pcs[mode][val_idx]
            for name, tmpl in clf_templates.items():
                clf_fold = clone(tmpl)
                clf_fold.fit(X_tr, y_tr)
                oof[(mode, name)][val_idx] = _predict_proba_3class(clf_fold, X_val)

    # ── NNLS stacking across all 27 candidates ───────────────────────────
    A = np.column_stack([oof[k].reshape(-1) for k in keys])
    b = np.eye(3)[y.astype(int)].reshape(-1)
    raw_w, _ = nnls(A, b)
    w_sum = raw_w.sum()
    K = len(keys)
    weights = {k: float(raw_w[i] / w_sum if w_sum > 0 else 1.0 / K)
               for i, k in enumerate(keys)}

    # ── Ensemble OOF probabilities ────────────────────────────────────────
    oof_ensemble = sum(weights[k] * oof[k] for k in keys)

    # ── Per-component metrics ─────────────────────────────────────────────
    metrics = []
    for k in keys:
        mode, name = k
        ll = log_loss(y, oof[k], labels=[0, 1, 2])
        try:
            auc = roc_auc_score(
                (y > 0).astype(int), oof[k][:, 1:].sum(axis=1)
            )
        except Exception:
            auc = np.nan
        metrics.append({"mode": mode, "method": name, "log_loss": ll,
                        "auc": auc, "weight": weights[k]})
    ll_ens = log_loss(y, oof_ensemble, labels=[0, 1, 2])
    auc_ens = roc_auc_score(
        (y > 0).astype(int), oof_ensemble[:, 1:].sum(axis=1)
    )
    metrics.append({"mode": "all", "method": "super_learner",
                    "log_loss": ll_ens, "auc": auc_ens, "weight": 1.0})
    metrics_df = pd.DataFrame(metrics)
    metrics_df["prl"] = prl(y, oof_ensemble)

    # ── Retrain on full data ──────────────────────────────────────────────
    fitted_clfs = {}
    for mode in FEATURE_MODES:
        X_full = X_pcs[mode]
        for name, tmpl in clf_templates.items():
            clf = clone(tmpl)
            clf.fit(X_full, y)
            fitted_clfs[(mode, name)] = clf

    return {
        "oof_proba":         oof_ensemble,
        "oof_per_model":     oof,
        "weights":           weights,
        "pipelines":         pipelines,
        "classifiers":       fitted_clfs,
        "component_metrics": metrics_df,
    }

## § 3  Main loop: train over 25 imputations

In [4]:
all_metrics = []

for cy in range(1, 6):
    for ud in range(1, 6):

        pkl_path = INTERIM / f"sl_model_{cy}_{ud}.pkl"

        # Resume: skip if already completed
        if pkl_path.exists():
            result = joblib.load(pkl_path)
            m = result["component_metrics"].copy()
            m["cy"], m["ud"] = cy, ud
            all_metrics.append(m)
            sl_row = m[m["method"] == "super_learner"].iloc[0]
            print(f"── CY {cy}/5, UD {ud}/5 ── (cached)  "
                  f"PRL={sl_row['prl']:.3f}  AUC={sl_row['auc']:.3f}")
            continue

        # ── Load data ──────────────────────────────────────────────────────
        dd = pd.read_parquet(INTERIM / f"dd_spat_{cy}_{ud}.parquet")
        dd = add_temporal_features(dd)

        train = dd[
            (dd["onset_A"] == 1) &
            (dd["intervention"].notna())
        ].copy()

        y = train["intervention"].astype(int).values

        # Pre-build W matrix cache for fixed-point burnout
        onset_mask = (dd["onset_A"] == 1)
        W_cache = _build_W_cache(dd, onset_mask)

        print(f"── CY {cy}/5, UD {ud}/5 ── n={len(train):,} onset dyads, "
              f"{(y==1).sum()} gov, {(y==2).sum()} opp")

        # ── Fixed-point iteration on spatial lags ─────────────────────────
        prev_spat = None
        for fp_iter in range(MAX_FP_ITER):
            result = train_super_learner(train, y, seed=SEED + cy * 10 + ud)

            p_gov = result["oof_proba"][:, 1]
            p_opp = result["oof_proba"][:, 2]
            train_updated = update_spatial_lags_proba(
                train, p_gov, p_opp, W_cache=W_cache
            )

            new_spat = train_updated[["spat_gov", "spat_opp"]].fillna(0).values
            if prev_spat is not None:
                delta = float(np.abs(new_spat - prev_spat).mean())
                print(f"   FP iter {fp_iter}: Δ spat = {delta:.5f}", end="")
                if delta < FP_TOL:
                    print("  ✓ converged")
                    break
                print()
            else:
                print(f"   FP iter {fp_iter}: initial")

            prev_spat = new_spat
            train = train_updated

        # ── Save outputs ──────────────────────────────────────────────────
        oof_df = train[["ddyear", "ccode_A", "ccode_B", "year", "intervention"]].copy()
        oof_df["p_none"] = result["oof_proba"][:, 0]
        oof_df["p_gov"]  = result["oof_proba"][:, 1]
        oof_df["p_opp"]  = result["oof_proba"][:, 2]
        oof_df["cy"], oof_df["ud"] = cy, ud
        oof_df.to_parquet(INTERIM / f"sl_oof_{cy}_{ud}.parquet", index=False)

        w_rows = [{"cy": cy, "ud": ud, "mode": k[0], "method": k[1], "weight": v}
                  for k, v in result["weights"].items()]
        w_df = pd.DataFrame(w_rows)
        w_df.to_parquet(INTERIM / f"sl_weights_{cy}_{ud}.parquet", index=False)

        joblib.dump(result, pkl_path)

        m = result["component_metrics"].copy()
        m["cy"], m["ud"] = cy, ud
        all_metrics.append(m)

        sl_row = m[m["method"] == "super_learner"].iloc[0]
        n_pca = {mode: p["n_pca"] for mode, p in result["pipelines"].items()}
        nonzero = sum(1 for v in result["weights"].values() if v > 0.005)
        print(f"   PRL={sl_row['prl']:.3f}  AUC={sl_row['auc']:.3f}  "
              f"n_PCA={n_pca}  {nonzero}/27 active components")

metrics_all = pd.concat(all_metrics, ignore_index=True)
metrics_all.to_parquet(INTERIM / "sl_cv_metrics.parquet", index=False)

print(f"\nDone. {len(all_metrics)} imputation draws completed.")

── CY 1/5, UD 1/5 ── (cached)  PRL=0.415  AUC=0.969


── CY 1/5, UD 2/5 ── n=25,872 onset dyads, 125 gov, 86 opp


   FP iter 0: initial


   FP iter 1: Δ spat = 0.00162


   FP iter 2: Δ spat = 0.00154


...

   FP iter 0: initial


   FP iter 1: Δ spat = 0.00175


   FP iter 2: Δ spat = 0.00190


   FP iter 3: Δ spat = 0.00109


   FP iter 4: Δ spat = 0.00120
   PRL=0.399  AUC=0.960  n_PCA={'X': 50, 'W': 13, 'XW': 59}  10/27 active components

Done. 25 imputation draws completed.


## § 3b  Fixed-point burnout

The main FP loop (§ 3) retrains the full super-learner at each iteration and
uses OOF predictions to update the spatial lags.  With `MAX_FP_ITER = 5`,
the mean absolute change in `spat_gov`/`spat_opp` at the end of training is
~0.006 — far above the 1e-4 tolerance.

The burnout phase applies the **frozen** fitted ensemble repeatedly, updating
spatial lags from full-model predictions until Δ < `BURNOUT_TOL`.  No
retraining occurs; each pass is seconds.  The converged lags are saved to
`sl_spat_conv_{cy}_{ud}.parquet` for use in nb06.

Note: burnout passes use full-model predictions, not OOF.  Each dyad's
contribution to its own spatial lag is diluted across ~190 co-conflict
interveners, so leakage is negligible.


In [5]:
# ── Fixed-point burnout ────────────────────────────────────────────────────
# Converge spatial lags with the frozen fitted model.
# Only the W and XW channels use spatial lags, but the full unified ensemble
# is applied at each iteration (X-channel predictions are constant).

import sys
sys.path.insert(0, str(Path.cwd() / "src"))
from shadow.data.spatial import _build_W_cache, update_spatial_lags_proba

MAX_BURNOUT = 20
BURNOUT_TOL = 5e-4  # achievable floor given RF prediction noise


def _predict_with_bundle(bundle: dict, df: pd.DataFrame) -> np.ndarray:
    """Apply the fitted unified super-learner ensemble to a DataFrame."""
    preds = np.zeros((len(df), 3))
    total_w = 0.0
    for (mode, name), clf in bundle["classifiers"].items():
        w = bundle["weights"].get((mode, name), 0.0)
        if w < 1e-8:
            continue
        pipe = bundle["pipelines"][mode]
        feat_cols = pipe["feat_cols"]
        X_raw = df[feat_cols].fillna(0).to_numpy(dtype=float)
        X_sc = pipe["scaler"].transform(X_raw)
        X_pc = pipe["pca"].transform(X_sc)
        preds += w * _predict_proba_3class(clf, X_pc)
        total_w += w
    return preds / total_w


print("── Fixed-point burnout ─────────────────────────────────────────────────")
for cy in range(1, 6):
    for ud in range(1, 6):
        conv_path = INTERIM / f"sl_spat_conv_{cy}_{ud}.parquet"
        if conv_path.exists():
            print(f"CY {cy}/5 UD {ud}/5  (cached)")
            continue

        pkl_path = INTERIM / f"sl_model_{cy}_{ud}.pkl"
        if not pkl_path.exists():
            print(f"CY {cy}/5 UD {ud}/5  model not yet trained, skipping")
            continue

        bundle = joblib.load(pkl_path)
        dd     = pd.read_parquet(INTERIM / f"dd_spat_{cy}_{ud}.parquet")
        dd     = add_temporal_features(dd)
        train  = dd[
            (dd["onset_A"] == 1) &
            (dd["intervention"].notna())
        ].copy()
        onset_mask = (dd["onset_A"] == 1)
        W_cache    = _build_W_cache(dd, onset_mask)

        current    = train.copy()
        prev_spat  = current[["spat_gov", "spat_opp"]].fillna(0).values

        deltas = []
        for bo in range(MAX_BURNOUT):
            proba   = _predict_with_bundle(bundle, current)
            updated = update_spatial_lags_proba(
                current, proba[:, 1], proba[:, 2], W_cache=W_cache
            )
            new_spat = updated[["spat_gov", "spat_opp"]].fillna(0).values
            delta    = float(np.abs(new_spat - prev_spat).mean())
            deltas.append(delta)
            if delta < BURNOUT_TOL:
                print(f"CY {cy}/5 UD {ud}/5  burnout converged iter={bo}  Δ={delta:.6f}")
                break
            prev_spat = new_spat
            current   = updated
        else:
            print(f"CY {cy}/5 UD {ud}/5  burnout max iter  Δ={delta:.6f}")

        # Save converged spatial lags (ddyear = join key for nb06)
        conv_df = current[["ddyear", "spat_gov", "spat_opp"]].rename(
            columns={"spat_gov": "spat_gov_conv", "spat_opp": "spat_opp_conv"}
        )
        conv_df["cy"], conv_df["ud"] = cy, ud
        conv_df["deltas"] = str(deltas)
        conv_df.to_parquet(conv_path, index=False)

print("Burnout complete.")

── Fixed-point burnout ─────────────────────────────────────────────────


CY 1/5 UD 1/5  burnout converged iter=3  Δ=0.000352


CY 1/5 UD 2/5  burnout converged iter=2  Δ=0.000318


CY 1/5 UD 3/5  burnout converged iter=3  Δ=0.000286


CY 1/5 UD 4/5  burnout converged iter=3  Δ=0.000468


...

CY 5/5 UD 1/5  burnout converged iter=3  Δ=0.000439


CY 5/5 UD 2/5  burnout converged iter=3  Δ=0.000484


CY 5/5 UD 3/5  burnout converged iter=3  Δ=0.000418


CY 5/5 UD 4/5  burnout converged iter=4  Δ=0.000447


CY 5/5 UD 5/5  burnout converged iter=3  Δ=0.000304
Burnout complete.


## § 4  Validation & summary table

In [6]:
# ── Summary: which (mode, method) pairs earn weight? ─────────────────────
name_map = {
    "rf":           "Random forest",
    "hgb":          "HGB (lr=0.10)",
    "hgb_lo":       "HGB (lr=0.05)",
    "ridge":        "Ridge logit",
    "glmnet":       "Elastic-net logit",
    "lasso":        "Lasso logit",
    "multinom":     "Multinomial logit",
    "mlp_sm":       "MLP (25,)",
    "mlp_lg":       "MLP (100,50)",
    "super_learner": "Super learner",
}

components = metrics_all[metrics_all["method"] != "super_learner"]
summary = (
    components
    .groupby(["mode", "method"])[["log_loss", "auc", "weight"]]
    .mean()
    .round(4)
    .sort_values("weight", ascending=False)
    .reset_index()
)
summary["method_label"] = summary["method"].map(name_map).fillna(summary["method"])

print(f"{'='*70}")
print("  Unified ensemble: mean weight by (mode, method) across draws")
print(f"{'='*70}")
# Show only components with mean weight > 0.5%
active = summary[summary["weight"] > 0.005]
for _, row in active.iterrows():
    print(f"  {row['mode']:>2s} × {row['method_label']:<20s}  "
          f"wt={row['weight']:.3f}  AUC={row['auc']:.3f}  LL={row['log_loss']:.4f}")

inactive = summary[summary["weight"] <= 0.005]
print(f"\n  ({len(inactive)} components with weight < 0.5%)")

# ── Overall super learner performance ─────────────────────────────────────
sl_only = metrics_all[metrics_all["method"] == "super_learner"]
print(f"\n{'='*70}")
print(f"  Super learner ensemble (27 candidates, unified NNLS)")
print(f"{'='*70}")
print(f"  PRL = {sl_only['prl'].mean():.3f} (SD {sl_only['prl'].std():.3f})")
print(f"  AUC = {sl_only['auc'].mean():.3f} (SD {sl_only['auc'].std():.3f})")
print(f"  n draws = {len(sl_only)}")

# ── Weight totals by feature mode ─────────────────────────────────────────
mode_totals = components.groupby(["cy", "ud", "mode"])["weight"].sum().reset_index()
mode_summary = mode_totals.groupby("mode")["weight"].agg(["mean", "std"]).round(3)
print(f"\n  Total weight by feature set:")
for mode, row in mode_summary.iterrows():
    print(f"    {mode:>2s}: {row['mean']:.3f} (SD {row['std']:.3f})")

print(f"\nOOF files: {len(list(INTERIM.glob('sl_oof_[0-9]_[0-9].parquet')))} "
      f"(expected 25)")

  Unified ensemble: mean weight by (mode, method) across draws
   X × Random forest         wt=0.299  AUC=0.941  LL=0.0497
  XW × Random forest         wt=0.229  AUC=0.946  LL=0.0458
   X × MLP (100,50)          wt=0.092  AUC=0.914  LL=0.0389
  XW × Multinomial logit     wt=0.087  AUC=0.942  LL=0.0371
  XW × MLP (100,50)          wt=0.087  AUC=0.893  LL=0.0409
  XW × HGB (lr=0.05)         wt=0.045  AUC=0.935  LL=0.0432
   X × HGB (lr=0.05)         wt=0.036  AUC=0.934  LL=0.0435
   W × Multinomial logit     wt=0.030  AUC=0.848  LL=0.0452
   W × Random forest         wt=0.020  AUC=0.884  LL=0.0723
   X × MLP (25,)             wt=0.019  AUC=0.801  LL=0.0579
  XW × MLP (25,)             wt=0.010  AUC=0.781  LL=0.0610
   X × HGB (lr=0.10)         wt=0.010  AUC=0.826  LL=0.1052
   W × HGB (lr=0.05)         wt=0.009  AUC=0.873  LL=0.0457
  XW × HGB (lr=0.10)         wt=0.007  AUC=0.804  LL=0.1295
   W × MLP (100,50)          wt=0.007  AUC=0.760  LL=0.0499
   W × Lasso logit           wt=0.005